# Intraday Data Recon

Pull 5-minute intraday bars from Bloomberg and yfinance, run source-level sanity checks, and reconcile the overlapping window between the two vendors.

Note: Yahoo Finance 5-minute history is usually limited to about the most recent 60 calendar days, so the reconciliation section makes the true overlap explicit even when Bloomberg covers the full 3-month request.


In [ ]:
from pathlib import Path
import datetime as dt
import re
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

DATA_UTILS_DIR = Path.cwd()
if not (DATA_UTILS_DIR / "blp_api.py").exists():
    candidate = Path.cwd() / "21_Backtest" / "draft_obs" / "utils" / "data_utils"
    if candidate.exists():
        DATA_UTILS_DIR = candidate
    else:
        raise FileNotFoundError("Could not locate utils/data_utils from the current working directory.")

if str(DATA_UTILS_DIR) not in sys.path:
    sys.path.append(str(DATA_UTILS_DIR))

CACHE_DIR = DATA_UTILS_DIR / "cache" / "intraday_data_recon"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SYMBOL_PAIRS = [
    {"label": "SPY", "bbg": "SPY US Equity", "yf": "SPY"},
]

BAR_INTERVAL_MINUTES = 5
LOOKBACK_DAYS = 90
YF_MAX_LOOKBACK_DAYS = 60
REFRESH_DATA = False

SESSION_START = dt.time(9, 30)
SESSION_END = dt.time(16, 0)
OHLCV_COLUMNS = ["open", "high", "low", "close", "volume"]
SOURCE_ORDER = ["bbg", "yfinance"]

END_TS = pd.Timestamp.now(tz="US/Eastern").floor("min")
START_TS = END_TS - pd.Timedelta(days=LOOKBACK_DAYS)
YF_START_TS = max(START_TS, END_TS - pd.Timedelta(days=YF_MAX_LOOKBACK_DAYS))
CACHE_TAG = END_TS.strftime("%Y%m%d")
DEFAULT_LABEL = SYMBOL_PAIRS[0]["label"]

print("Data utils dir :", DATA_UTILS_DIR)
print("Cache dir      :", CACHE_DIR)
print("Bloomberg      :", f"{START_TS} -> {END_TS}")
print("yfinance floor :", f"{YF_START_TS} -> {END_TS}")
print("Symbols        :", SYMBOL_PAIRS)


In [ ]:
from blp_api import BLPServer


def empty_intraday_frame() -> pd.DataFrame:
    return pd.DataFrame(columns=OHLCV_COLUMNS)


def safe_slug(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()


def cache_path(label: str, source: str) -> Path:
    return CACHE_DIR / f"{safe_slug(label)}_{source}_{BAR_INTERVAL_MINUTES}m_{LOOKBACK_DAYS}d_{CACHE_TAG}.pkl"


def _configure_yfinance_cache(yf, cache_dir: Path) -> None:
    try:
        import yfinance.cache as yf_cache

        if hasattr(yf, "set_tz_cache_location"):
            yf.set_tz_cache_location(str(cache_dir))

        yf_cache._TzCacheManager._tz_cache = yf_cache._TzCacheDummy()
        yf_cache._CookieCacheManager._Cookie_cache = yf_cache._CookieCacheDummy()
        if hasattr(yf_cache, "_ISINCacheManager") and hasattr(yf_cache, "_ISINCacheDummy"):
            yf_cache._ISINCacheManager._isin_cache = yf_cache._ISINCacheDummy()
    except Exception as exc:
        print(f"[warn] Unable to reconfigure yfinance cache: {exc}")


def normalize_intraday_frame(df: pd.DataFrame, assume_tz: str) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return empty_intraday_frame()

    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = out.columns.get_level_values(0)
    out.columns = [str(col).lower() for col in out.columns]

    if "time" in out.columns:
        out = out.set_index("time")

    if not isinstance(out.index, pd.DatetimeIndex):
        out.index = pd.to_datetime(out.index)

    out.index = pd.to_datetime(out.index)
    if out.index.tz is None:
        out.index = out.index.tz_localize(assume_tz)
    out.index = out.index.tz_convert("US/Eastern")

    keep = [col for col in OHLCV_COLUMNS if col in out.columns]
    out = out[keep].sort_index()
    out = out[~out.index.duplicated(keep="last")]

    for col in keep:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    price_cols = [col for col in ["open", "high", "low", "close"] if col in out.columns]
    if price_cols:
        out = out.dropna(subset=price_cols)

    for col in OHLCV_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan

    return out[OHLCV_COLUMNS]


def keep_regular_session(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    session_mask = (df.index.time >= SESSION_START) & (df.index.time < SESSION_END)
    return df.loc[session_mask].copy()


def fetch_bbg_intraday(ticker: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    query_start = start.tz_convert("US/Eastern").tz_localize(None).to_pydatetime()
    query_end = end.tz_convert("US/Eastern").tz_localize(None).to_pydatetime()
    with BLPServer() as blp:
        raw = blp.BLPGetIntradayBar(
            security=ticker,
            start=query_start,
            end=query_end,
            event_type="trade",
            interval=BAR_INTERVAL_MINUTES,
        )
    frame = normalize_intraday_frame(raw, assume_tz="US/Eastern")
    frame = keep_regular_session(frame)
    return frame.loc[(frame.index >= start) & (frame.index <= end)]


def fetch_yfinance_intraday(ticker: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    import yfinance as yf

    tz_cache_dir = CACHE_DIR / "yfinance_tz_cache"
    tz_cache_dir.mkdir(parents=True, exist_ok=True)
    _configure_yfinance_cache(yf, tz_cache_dir)

    requested_days = max(1, int(np.ceil((end - start) / pd.Timedelta(days=1))))
    if requested_days > YF_MAX_LOOKBACK_DAYS:
        effective_start = max(start, end - pd.Timedelta(days=YF_MAX_LOOKBACK_DAYS))
        warnings.warn(
            f"Requested {requested_days} calendar days of 5-minute data for {ticker}; "
            f"yfinance typically exposes about {YF_MAX_LOOKBACK_DAYS} days. "
            f"Using overlap window starting {effective_start.date()}.",
            stacklevel=2,
        )
        raw = yf.download(
            ticker,
            period=f"{YF_MAX_LOOKBACK_DAYS}d",
            interval="5m",
            auto_adjust=False,
            progress=False,
            prepost=False,
        )
    else:
        effective_start = start
        raw = yf.download(
            ticker,
            start=effective_start.strftime("%Y-%m-%d"),
            end=(end + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
            interval="5m",
            auto_adjust=False,
            progress=False,
            prepost=False,
        )

    frame = normalize_intraday_frame(raw, assume_tz="UTC")
    frame = keep_regular_session(frame)
    return frame.loc[(frame.index >= effective_start) & (frame.index <= end)]


def load_or_fetch_source(label: str, source: str, ticker: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    path = cache_path(label, source)
    if path.exists() and not REFRESH_DATA:
        print(f"[{label}] {source}: loading cache from {path.name}")
        return pd.read_pickle(path)

    fetcher = fetch_bbg_intraday if source == "bbg" else fetch_yfinance_intraday
    frame = fetcher(ticker, start, end)
    frame.to_pickle(path)
    print(f"[{label}] {source}: cached {len(frame):,} bars -> {path.name}")
    return frame


In [ ]:
def prepare_source_frame(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    if df.empty:
        columns = OHLCV_COLUMNS + [
            "bar_ret", "abs_bar_ret", "log_volume", "trade_date", "hhmm",
            "weekday", "month", "bar_idx", "source"
        ]
        return pd.DataFrame(columns=columns, index=pd.DatetimeIndex([], tz="US/Eastern"))

    out = df.copy().sort_index()
    local_index = out.index.tz_convert("US/Eastern")
    out["trade_date"] = pd.Index(local_index.date)
    out["bar_idx"] = out.groupby("trade_date").cumcount()
    out["hhmm"] = local_index.hour * 100 + local_index.minute
    out["bar_ret"] = out.groupby("trade_date")["close"].pct_change().fillna(0.0)
    out["abs_bar_ret"] = out["bar_ret"].abs()
    out["log_volume"] = np.log(out["volume"].clip(lower=1.0))
    out["weekday"] = pd.Index(out["trade_date"]).map(lambda d: pd.Timestamp(d).day_name())
    out["month"] = pd.Index(out["trade_date"]).map(lambda d: pd.Timestamp(d).strftime("%Y-%m"))
    out["source"] = source_name
    return out


def coverage_summary(frames_by_source: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for source_name, frame in frames_by_source.items():
        if frame.empty:
            rows.append({
                "source": source_name,
                "n_rows": 0,
                "n_trade_dates": 0,
                "start_ts": pd.NaT,
                "end_ts": pd.NaT,
                "min_bars_per_date": np.nan,
                "median_bars_per_date": np.nan,
                "max_bars_per_date": np.nan,
                "zero_volume_share": np.nan,
            })
            continue

        session_counts = frame.groupby("trade_date").size()
        rows.append({
            "source": source_name,
            "n_rows": len(frame),
            "n_trade_dates": frame["trade_date"].nunique(),
            "start_ts": frame.index.min(),
            "end_ts": frame.index.max(),
            "min_bars_per_date": int(session_counts.min()),
            "median_bars_per_date": float(session_counts.median()),
            "max_bars_per_date": int(session_counts.max()),
            "zero_volume_share": float((frame["volume"] == 0).mean()),
        })
    return pd.DataFrame(rows)


def sample_trade_dates(trade_dates, n: int = 2) -> list:
    dates = list(pd.Index(sorted(pd.unique(trade_dates))))
    if not dates:
        return []
    picks = [dates[0], dates[len(dates) // 2], dates[-1]]
    unique_picks = []
    for trade_date in picks:
        if trade_date not in unique_picks:
            unique_picks.append(trade_date)
    return unique_picks[:n]


def plot_source_sanity(frame: pd.DataFrame, label: str, source_name: str) -> None:
    if frame.empty:
        print(f"[{label}] {source_name}: no data available for sanity plots.")
        return

    session_counts = frame.groupby("trade_date").size()
    intraday_profile = frame.groupby("hhmm").agg(
        bars=("close", "size"),
        avg_volume=("volume", "mean"),
        avg_abs_bar_ret=("abs_bar_ret", "mean"),
    )
    monthly_counts = frame.groupby("month").size()
    weekday_counts = frame.groupby("weekday").size().reindex(
        ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    ).fillna(0)

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    axes[0, 0].plot(pd.to_datetime(session_counts.index), session_counts.values, marker="o", linewidth=1.25)
    axes[0, 0].set_title(f"{label} / {source_name}: bars per trade date")
    axes[0, 0].set_ylabel("bars")
    axes[0, 0].tick_params(axis="x", rotation=45)

    axes[0, 1].bar(monthly_counts.index, monthly_counts.values, color="#4C78A8")
    axes[0, 1].set_title(f"{label} / {source_name}: observations by month")
    axes[0, 1].tick_params(axis="x", rotation=45)

    axes[1, 0].plot(intraday_profile.index, intraday_profile["avg_volume"], color="#F58518")
    axes[1, 0].set_title(f"{label} / {source_name}: average volume by intraday period")
    axes[1, 0].set_xlabel("HHMM")
    axes[1, 0].set_ylabel("avg volume")

    axes[1, 1].plot(intraday_profile.index, intraday_profile["avg_abs_bar_ret"], color="#54A24B")
    axes[1, 1].set_title(f"{label} / {source_name}: average absolute return by intraday period")
    axes[1, 1].set_xlabel("HHMM")
    axes[1, 1].set_ylabel("avg |bar return|")

    plt.tight_layout()
    plt.show()

    display(session_counts.describe().to_frame("bars_per_trade_date").T)
    display(weekday_counts.to_frame("obs"))


def plot_sample_days(frame: pd.DataFrame, label: str, source_name: str, n: int = 2) -> None:
    if frame.empty:
        return

    for trade_date in sample_trade_dates(frame["trade_date"], n=n):
        day = frame[frame["trade_date"] == trade_date].copy()
        display(day[OHLCV_COLUMNS].describe().T)

        fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
        axes[0].plot(day.index, day["close"], color="#4C78A8")
        axes[0].set_title(f"{label} / {source_name}: trade date {trade_date} - close")
        axes[0].set_ylabel("price")

        axes[1].bar(day.index, day["volume"], width=0.003, color="#F58518")
        axes[1].set_title("Volume")
        axes[1].set_ylabel("shares")

        axes[2].plot(day.index, day["bar_ret"], color="#54A24B")
        axes[2].axhline(0.0, color="black", linewidth=0.8)
        axes[2].set_title("5-minute returns")
        axes[2].set_ylabel("return")
        axes[2].set_xlabel("timestamp")

        plt.tight_layout()
        plt.show()


In [ ]:
def align_sources(left: pd.DataFrame, right: pd.DataFrame, left_name: str = "bbg", right_name: str = "yfinance") -> pd.DataFrame:
    if left.empty or right.empty:
        return pd.DataFrame()

    keep_cols = ["open", "high", "low", "close", "volume", "bar_ret", "trade_date", "bar_idx", "hhmm"]
    overlap = left[keep_cols].join(right[keep_cols], how="inner", lsuffix=f"_{left_name}", rsuffix=f"_{right_name}")
    overlap = overlap.sort_index()
    if overlap.empty:
        return overlap

    price_ref = overlap[f"close_{left_name}"].replace(0, np.nan)
    volume_ref = overlap[f"volume_{left_name}"].replace(0, np.nan)

    for field in ["open", "high", "low", "close"]:
        overlap[f"{field}_diff"] = overlap[f"{field}_{left_name}"] - overlap[f"{field}_{right_name}"]
        overlap[f"{field}_abs_diff"] = overlap[f"{field}_diff"].abs()
        overlap[f"{field}_diff_bps"] = 1e4 * overlap[f"{field}_diff"] / price_ref

    overlap["volume_diff"] = overlap[f"volume_{left_name}"] - overlap[f"volume_{right_name}"]
    overlap["volume_abs_diff"] = overlap["volume_diff"].abs()
    overlap["volume_diff_pct"] = overlap["volume_diff"] / volume_ref
    overlap["bar_ret_diff_bps"] = 1e4 * (
        overlap[f"bar_ret_{left_name}"] - overlap[f"bar_ret_{right_name}"]
    )
    overlap["same_return_sign"] = (
        np.sign(overlap[f"bar_ret_{left_name}"]) == np.sign(overlap[f"bar_ret_{right_name}"])
    )
    return overlap


def reconciliation_summary(
    label: str,
    left: pd.DataFrame,
    right: pd.DataFrame,
    overlap: pd.DataFrame,
    left_name: str = "bbg",
    right_name: str = "yfinance",
) -> pd.Series:
    if overlap.empty:
        return pd.Series({
            "label": label,
            f"rows_{left_name}": len(left),
            f"rows_{right_name}": len(right),
            "rows_overlap": 0,
            f"trade_dates_{left_name}": left['trade_date'].nunique() if not left.empty else 0,
            f"trade_dates_{right_name}": right['trade_date'].nunique() if not right.empty else 0,
            "trade_dates_overlap": 0,
            "overlap_share_vs_bbg": 0.0,
            "overlap_share_vs_yfinance": 0.0,
            "median_close_diff_bps": np.nan,
            "p95_close_diff_bps": np.nan,
            "max_close_diff_bps": np.nan,
            "median_bar_ret_diff_bps": np.nan,
            "p95_bar_ret_diff_bps": np.nan,
            "median_volume_diff_pct": np.nan,
            "p95_volume_diff_pct": np.nan,
            "same_return_sign_share": np.nan,
            "overlap_start": pd.NaT,
            "overlap_end": pd.NaT,
        })

    overlap_trade_dates = overlap[f"trade_date_{left_name}"].nunique()
    return pd.Series({
        "label": label,
        f"rows_{left_name}": len(left),
        f"rows_{right_name}": len(right),
        "rows_overlap": len(overlap),
        f"trade_dates_{left_name}": left['trade_date'].nunique(),
        f"trade_dates_{right_name}": right['trade_date'].nunique(),
        "trade_dates_overlap": overlap_trade_dates,
        "overlap_share_vs_bbg": len(overlap) / len(left) if len(left) else np.nan,
        "overlap_share_vs_yfinance": len(overlap) / len(right) if len(right) else np.nan,
        "median_close_diff_bps": overlap["close_diff_bps"].abs().median(),
        "p95_close_diff_bps": overlap["close_diff_bps"].abs().quantile(0.95),
        "max_close_diff_bps": overlap["close_diff_bps"].abs().max(),
        "median_bar_ret_diff_bps": overlap["bar_ret_diff_bps"].abs().median(),
        "p95_bar_ret_diff_bps": overlap["bar_ret_diff_bps"].abs().quantile(0.95),
        "median_volume_diff_pct": overlap["volume_diff_pct"].abs().median(),
        "p95_volume_diff_pct": overlap["volume_diff_pct"].abs().quantile(0.95),
        "same_return_sign_share": overlap["same_return_sign"].mean(),
        "overlap_start": overlap.index.min(),
        "overlap_end": overlap.index.max(),
    })


def daily_bar_count_comparison(
    left: pd.DataFrame,
    right: pd.DataFrame,
    overlap: pd.DataFrame,
    left_name: str = "bbg",
    right_name: str = "yfinance",
) -> pd.DataFrame:
    left_counts = left.groupby("trade_date").size().rename(f"bars_{left_name}") if not left.empty else pd.Series(dtype=int)
    right_counts = right.groupby("trade_date").size().rename(f"bars_{right_name}") if not right.empty else pd.Series(dtype=int)
    if overlap.empty:
        shared_counts = pd.Series(dtype=int, name="shared_bars")
    else:
        shared_counts = overlap.groupby(f"trade_date_{left_name}").size().rename("shared_bars")

    counts = pd.concat([left_counts, right_counts, shared_counts], axis=1).fillna(0).astype(int)
    if counts.empty:
        return counts

    counts["left_only_bars"] = counts[f"bars_{left_name}"] - counts["shared_bars"]
    counts["right_only_bars"] = counts[f"bars_{right_name}"] - counts["shared_bars"]
    counts["bar_count_diff"] = counts[f"bars_{left_name}"] - counts[f"bars_{right_name}"]
    return counts.sort_index()


def average_matched_summary(overlap: pd.DataFrame, left_name: str = "bbg", right_name: str = "yfinance") -> pd.DataFrame:
    if overlap.empty:
        return pd.DataFrame()

    rows = []
    for field in ["open", "high", "low", "close"]:
        rows.append({
            "metric": field,
            f"avg_{left_name}": overlap[f"{field}_{left_name}"].mean(),
            f"avg_{right_name}": overlap[f"{field}_{right_name}"].mean(),
            "avg_abs_diff": overlap[f"{field}_abs_diff"].mean(),
            "median_abs_diff": overlap[f"{field}_abs_diff"].median(),
            "avg_abs_diff_bps": overlap[f"{field}_diff_bps"].abs().mean(),
        })

    rows.extend([
        {
            "metric": "volume",
            f"avg_{left_name}": overlap[f"volume_{left_name}"].mean(),
            f"avg_{right_name}": overlap[f"volume_{right_name}"].mean(),
            "avg_abs_diff": overlap["volume_abs_diff"].mean(),
            "median_abs_diff": overlap["volume_abs_diff"].median(),
            "avg_abs_diff_bps": overlap["volume_diff_pct"].abs().mean() * 1e4,
        },
        {
            "metric": "bar_ret",
            f"avg_{left_name}": overlap[f"bar_ret_{left_name}"].mean(),
            f"avg_{right_name}": overlap[f"bar_ret_{right_name}"].mean(),
            "avg_abs_diff": overlap["bar_ret_diff_bps"].abs().mean() / 1e4,
            "median_abs_diff": overlap["bar_ret_diff_bps"].abs().median() / 1e4,
            "avg_abs_diff_bps": overlap["bar_ret_diff_bps"].abs().mean(),
        },
    ])

    result = pd.DataFrame(rows).set_index("metric")
    return result


def average_matched_profile(overlap: pd.DataFrame, left_name: str = "bbg", right_name: str = "yfinance") -> pd.DataFrame:
    if overlap.empty:
        return pd.DataFrame()

    hhmm_col = f"hhmm_{left_name}"
    profile = overlap.groupby(hhmm_col).agg(
        avg_close_bbg=(f"close_{left_name}", "mean"),
        avg_close_yfinance=(f"close_{right_name}", "mean"),
        avg_volume_bbg=(f"volume_{left_name}", "mean"),
        avg_volume_yfinance=(f"volume_{right_name}", "mean"),
        avg_abs_close_diff_bps=("close_diff_bps", lambda s: s.abs().mean()),
        avg_abs_bar_ret_diff_bps=("bar_ret_diff_bps", lambda s: s.abs().mean()),
        avg_abs_volume_diff_pct=("volume_diff_pct", lambda s: s.abs().mean()),
        matched_bars=(f"close_{left_name}", "size"),
    )
    return profile


def plot_abs_diff_report(overlap: pd.DataFrame, label: str, left_name: str = "bbg", right_name: str = "yfinance") -> None:
    if overlap.empty:
        print(f"[{label}] No overlapping timestamps to build abs-diff report.")
        return

    profile = average_matched_profile(overlap, left_name=left_name, right_name=right_name)
    rolling_window = min(78, max(5, len(overlap) // 10))
    abs_close_bps = overlap["close_diff_bps"].abs()
    abs_ret_bps = overlap["bar_ret_diff_bps"].abs()

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    axes[0, 0].plot(overlap.index, abs_close_bps, color="#E45756", alpha=0.35, label="abs close diff")
    axes[0, 0].plot(overlap.index, abs_close_bps.rolling(rolling_window, min_periods=1).mean(), color="#4C78A8", linewidth=1.5, label=f"{rolling_window}-bar mean")
    axes[0, 0].set_title(f"{label}: absolute close difference (bps)")
    axes[0, 0].set_ylabel("bps")
    axes[0, 0].legend(loc="best")

    axes[0, 1].plot(overlap.index, abs_ret_bps, color="#54A24B", alpha=0.35, label="abs return diff")
    axes[0, 1].plot(overlap.index, abs_ret_bps.rolling(rolling_window, min_periods=1).mean(), color="#F58518", linewidth=1.5, label=f"{rolling_window}-bar mean")
    axes[0, 1].set_title(f"{label}: absolute return difference (bps)")
    axes[0, 1].set_ylabel("bps")
    axes[0, 1].legend(loc="best")

    axes[1, 0].plot(profile.index, profile["avg_abs_close_diff_bps"], color="#E45756")
    axes[1, 0].set_title(f"{label}: average abs close diff by HHMM")
    axes[1, 0].set_xlabel("HHMM")
    axes[1, 0].set_ylabel("bps")

    axes[1, 1].plot(profile.index, profile["avg_abs_volume_diff_pct"] * 100.0, color="#72B7B2")
    axes[1, 1].set_title(f"{label}: average abs volume diff by HHMM")
    axes[1, 1].set_xlabel("HHMM")
    axes[1, 1].set_ylabel("%")

    plt.tight_layout()
    plt.show()


def plot_average_matched_result(overlap: pd.DataFrame, label: str, left_name: str = "bbg", right_name: str = "yfinance") -> None:
    if overlap.empty:
        print(f"[{label}] No overlapping timestamps to plot average matched result.")
        return

    profile = average_matched_profile(overlap, left_name=left_name, right_name=right_name)
    fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)

    axes[0].plot(profile.index, profile["avg_close_bbg"], label=left_name.upper(), color="#4C78A8")
    axes[0].plot(profile.index, profile["avg_close_yfinance"], label=right_name.title(), color="#F58518", alpha=0.9)
    axes[0].set_title(f"{label}: average matched close by HHMM")
    axes[0].set_ylabel("price")
    axes[0].legend(loc="best")

    axes[1].plot(profile.index, profile["avg_volume_bbg"], label=left_name.upper(), color="#4C78A8")
    axes[1].plot(profile.index, profile["avg_volume_yfinance"], label=right_name.title(), color="#F58518", alpha=0.9)
    axes[1].set_title(f"{label}: average matched volume by HHMM")
    axes[1].set_xlabel("HHMM")
    axes[1].set_ylabel("shares")
    axes[1].legend(loc="best")

    plt.tight_layout()
    plt.show()


def plot_reconciliation_days(overlap: pd.DataFrame, label: str, left_name: str = "bbg", right_name: str = "yfinance", n: int = 2) -> None:
    if overlap.empty:
        print(f"[{label}] No overlapping timestamps to reconcile.")
        return

    trade_date_col = f"trade_date_{left_name}"
    for trade_date in sample_trade_dates(overlap[trade_date_col], n=n):
        day = overlap[overlap[trade_date_col] == trade_date].copy()
        display(day[[
            f"close_{left_name}", f"close_{right_name}", "close_diff_bps",
            f"volume_{left_name}", f"volume_{right_name}", "volume_diff_pct",
            "bar_ret_diff_bps",
        ]].describe().T)

        fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

        axes[0].plot(day.index, day[f"close_{left_name}"], label=left_name.upper(), color="#4C78A8")
        axes[0].plot(day.index, day[f"close_{right_name}"], label=right_name.title(), color="#F58518", alpha=0.85)
        axes[0].set_title(f"{label}: overlapping close by source on {trade_date}")
        axes[0].set_ylabel("price")
        axes[0].legend(loc="best")

        axes[1].bar(day.index, day["close_diff_bps"], width=0.003, color="#E45756")
        axes[1].axhline(0.0, color="black", linewidth=0.8)
        axes[1].set_title("Close difference (bps, Bloomberg reference)")
        axes[1].set_ylabel("bps")

        axes[2].bar(day.index, day["volume_diff_pct"].fillna(0.0) * 100.0, width=0.003, color="#72B7B2")
        axes[2].axhline(0.0, color="black", linewidth=0.8)
        axes[2].set_title("Volume difference (% of Bloomberg volume)")
        axes[2].set_ylabel("%")
        axes[2].set_xlabel("timestamp")

        plt.tight_layout()
        plt.show()


In [ ]:
raw_data: dict[str, dict[str, pd.DataFrame]] = {}
prepared_data: dict[str, dict[str, pd.DataFrame]] = {}

for pair in SYMBOL_PAIRS:
    label = pair["label"]
    print(f"\n=== {label} ===")
    raw_data[label] = {}
    prepared_data[label] = {}

    for source_name, ticker in [("bbg", pair["bbg"]), ("yfinance", pair["yf"])]:
        try:
            frame = load_or_fetch_source(label, source_name, ticker, START_TS, END_TS)
        except Exception as exc:
            print(f"[{label}] {source_name} failed: {exc}")
            frame = empty_intraday_frame()

        raw_data[label][source_name] = frame
        prepared_data[label][source_name] = prepare_source_frame(frame, source_name)

        prepared = prepared_data[label][source_name]
        print(
            f"[{label}] {source_name}: rows={len(prepared):,}, "
            f"trade_dates={prepared['trade_date'].nunique() if not prepared.empty else 0}, "
            f"start={prepared.index.min() if not prepared.empty else 'NA'}, "
            f"end={prepared.index.max() if not prepared.empty else 'NA'}"
        )


In [ ]:
coverage_frames = []
for label, frames_by_source in prepared_data.items():
    summary = coverage_summary(frames_by_source)
    summary.insert(0, "label", label)
    coverage_frames.append(summary)

coverage_df = pd.concat(coverage_frames, ignore_index=True) if coverage_frames else pd.DataFrame()
coverage_df


In [ ]:
VIEW_LABEL = DEFAULT_LABEL

for source_name in SOURCE_ORDER:
    frame = prepared_data[VIEW_LABEL][source_name]
    plot_source_sanity(frame, VIEW_LABEL, source_name)
    plot_sample_days(frame, VIEW_LABEL, source_name, n=2)


In [ ]:
recon_tables: dict[str, pd.DataFrame] = {}
daily_count_tables: dict[str, pd.DataFrame] = {}
recon_rows = []

for label, frames_by_source in prepared_data.items():
    overlap = align_sources(frames_by_source["bbg"], frames_by_source["yfinance"])
    recon_tables[label] = overlap
    daily_count_tables[label] = daily_bar_count_comparison(
        frames_by_source["bbg"],
        frames_by_source["yfinance"],
        overlap,
    )
    recon_rows.append(
        reconciliation_summary(
            label,
            frames_by_source["bbg"],
            frames_by_source["yfinance"],
            overlap,
        )
    )

recon_summary_df = pd.DataFrame(recon_rows)
recon_summary_df


In [ ]:
overlap = recon_tables[VIEW_LABEL]
daily_counts = daily_count_tables[VIEW_LABEL]

if daily_counts.empty:
    print(f"[{VIEW_LABEL}] No daily bar count comparison available.")
else:
    display(daily_counts.tail(15))
    mismatched_days = daily_counts[daily_counts["bar_count_diff"] != 0]
    display(mismatched_days.head(15))

if overlap.empty:
    print(f"[{VIEW_LABEL}] No overlapping timestamps between Bloomberg and yfinance.")
else:
    avg_match = average_matched_summary(overlap)
    avg_profile = average_matched_profile(overlap)

    display(overlap[[
        "close_bbg", "close_yfinance", "close_diff_bps",
        "volume_bbg", "volume_yfinance", "volume_diff_pct",
        "bar_ret_bbg", "bar_ret_yfinance", "bar_ret_diff_bps",
    ]].head())
    display(avg_match)
    display(avg_profile.head(15))
    display(
        overlap.nlargest(15, "close_abs_diff")[[
            "close_bbg", "close_yfinance", "close_diff", "close_diff_bps",
            "volume_bbg", "volume_yfinance", "volume_diff_pct",
            "bar_ret_bbg", "bar_ret_yfinance", "bar_ret_diff_bps",
        ]]
    )
    plot_abs_diff_report(overlap, VIEW_LABEL)
    plot_average_matched_result(overlap, VIEW_LABEL)
    plot_reconciliation_days(overlap, VIEW_LABEL, n=2)
